In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from PIL import Image, ImageTk
import os
import json


LOGIN_FILE = "login_data.json"

def load_logins():
    if os.path.exists(LOGIN_FILE):
        with open(LOGIN_FILE, "r") as f:
            return json.load(f)
    return {}

def save_logins(logins):
    with open(LOGIN_FILE, "w") as f:
        json.dump(logins, f)

class LoginPage:
    def __init__(self, root):
        self.root = root
        self.root.title("Login - Soil Fertility Analysis")
        self.root.geometry("1000x700")
        self.bg_image_original = None
        self.bg_photo = None

        self.login_records = load_logins()

        self.bg_label = tk.Label(root)
        self.bg_label.place(x=0, y=0, relwidth=1, relheight=1)

        if os.path.exists("soillab.jpg"):
            try:
                self.bg_image_original = Image.open("soillab.jpg")
                self.update_background()
                self.root.bind("<Configure>", self.on_resize)
            except Exception as e:
                print(f"Could not load background image: {e}")

        self.title_label = tk.Label(root, text="Soil Fertility Analysis", 
                                    font=("Arial", 28, "bold"), fg="white", bg="#2e7d32")
        self.title_label.place(relx=0.5, rely=0.1, anchor="center")

        self.frame = tk.Frame(root, bg="#ffffff", bd=2)
        self.frame.place(relx=0.5, rely=0.5, anchor=tk.CENTER)

        title = tk.Label(self.frame, text="Login", font=("Arial", 18, "bold"), bg="white", fg="#333")
        title.grid(row=0, columnspan=2, pady=(10, 20))

        tk.Label(self.frame, text="Email:", font=("Arial", 12), bg="white").grid(row=1, column=0, padx=10, pady=10)
        self.email_entry = tk.Entry(self.frame, font=("Arial", 12), width=30)
        self.email_entry.grid(row=1, column=1, padx=10)

        tk.Label(self.frame, text="Password:", font=("Arial", 12), bg="white").grid(row=2, column=0, padx=10, pady=10)
        self.password_entry = tk.Entry(self.frame, show="*", font=("Arial", 12), width=30)
        self.password_entry.grid(row=2, column=1, padx=10)

        login_btn = tk.Button(self.frame, text="Login", font=("Arial", 12, "bold"), bg="#4CAF50", fg="white",
                              command=self.validate_login, width=20, height=2)
        login_btn.grid(row=3, columnspan=2, pady=20)

    def update_background(self):
        if self.bg_image_original and self.bg_label.winfo_exists():
            width = self.root.winfo_width()
            height = self.root.winfo_height()
            resized = self.bg_image_original.resize((width, height), Image.Resampling.LANCZOS)
            self.bg_photo = ImageTk.PhotoImage(resized)
            self.bg_label.configure(image=self.bg_photo)
            self.bg_label.image = self.bg_photo  

    def on_resize(self, event):
        self.update_background()

    def validate_login(self):
        email = self.email_entry.get().strip()
        password = self.password_entry.get().strip()

        if not email or not password:
            messagebox.showwarning("Input Error", "Please enter both email and password.")
            return

        if email in self.login_records:
            if self.login_records[email] == password:
                messagebox.showinfo("Login Success", f"Welcome back, {email}!")
            else:
                messagebox.showerror("Login Failed", "Password incorrect for this email.")
                return
        else:
            self.login_records[email] = password
            save_logins(self.login_records)
            messagebox.showinfo("Account Created", f"New account created for {email}")

        self.root.unbind("<Configure>")
        for widget in self.root.winfo_children():
            widget.destroy()

        SoilDataApp(self.root)


class SoilDataApp:
    def __init__(self, root):
        self.root = root
        self.data = None
        self.bg_image_original = None
        self.bg_photo = None

        self.root.title("Soil Data Visualizer")

        self.bg_label = tk.Label(self.root)
        self.bg_label.place(x=0, y=0, relwidth=1, relheight=1)

        if os.path.exists("soil-fertility.jpg"):
            try:
                self.bg_image_original = Image.open("soil-fertility.jpg")
                self.update_background()
                self.root.bind("<Configure>", self.on_resize)
            except Exception as e:
                print(f"Could not load background image: {e}")
        else:
            print("soil-fertility.jpg not found!")

        self.logout_btn = tk.Button(root, text="Logout", command=self.logout,
                                    font=("Arial", 12, "bold"), bg="#f44336", fg="white", width=12, height=2)
        self.logout_btn.place(relx=0.98, rely=0.05, anchor="ne")

        self.load_btn = tk.Button(root, text="Load CSV Dataset", command=self.load_data,
                                  font=("Arial", 14, "bold"), bg="#4CAF50", fg="white", width=20, height=2)
        self.load_btn.place(x=40, y=30)

        graph_label = tk.Label(root, text="Select column to visualize:", bg="#ffffff",
                               font=("Arial", 14, "bold"))
        graph_label.place(x=40, y=110)

        columns = ['pH', 'EC', 'Sand', 'Clay', 'Output']
        self.graph_buttons = []

        for idx, col in enumerate(columns):
            btn = tk.Button(root, text=f"Graph: {col}", width=18, height=2,
                            command=lambda c=col: self.open_graph_window(c),
                            font=("Arial", 12), bg="#2196F3", fg="white")
            btn.place(x=40 + (idx * 180), y=160)
            self.graph_buttons.append(btn)

    def update_background(self):
        if self.bg_image_original:
            width = self.root.winfo_width()
            height = self.root.winfo_height()
            resized = self.bg_image_original.resize((width, height), Image.Resampling.LANCZOS)
            self.bg_photo = ImageTk.PhotoImage(resized)
            self.bg_label.configure(image=self.bg_photo)
            self.bg_label.image = self.bg_photo  

    def on_resize(self, event):
        self.update_background()
        self.logout_btn.place(relx=0.98, rely=0.05, anchor="ne")

    def load_data(self):
        file_path = filedialog.askopenfilename(filetypes=[("CSV Files", "*.csv")])
        if file_path:
            try:
                self.data = pd.read_csv(file_path)
                self.open_data_window()
            except Exception as e:
                messagebox.showerror("Error", f"Could not load dataset:\n{e}")

    def open_data_window(self):
        if self.data is None:
            return

        data_win = tk.Toplevel(self.root)
        data_win.title("Dataset Preview")
        data_win.geometry("900x400")

        title = tk.Label(data_win, text="Top 10 Rows from Dataset", font=("Arial", 14, "bold"))
        title.pack(pady=10)

        text_area = tk.Text(data_win, height=15, width=110, font=("Courier", 10), bg="#f4f4f4")
        text_area.pack(padx=20)
        text_area.insert(tk.END, self.data.head(10).to_string(index=False))

    def open_graph_window(self, column):
        if self.data is None:
            messagebox.showwarning("No Data", "Please load a CSV file first.")
            return

        if column not in self.data.columns:
            messagebox.showerror("Column Error", f"'{column}' column not found in dataset.")
            return

        graph_win = tk.Toplevel(self.root)
        graph_win.title(f"Graph - {column}")
        graph_win.geometry("900x500")

        type_counts = self.data[column].value_counts().head(10)
        x = type_counts.index.astype(str)
        y = type_counts.values

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(x, y, color='green')
        ax.set_title(f"Top 10 Frequencies in '{column}'", fontsize=12)
        ax.set_xlabel("Value")
        ax.set_ylabel("Count")
        ax.tick_params(axis='x', rotation=15)
        plt.tight_layout()

        canvas = FigureCanvasTkAgg(fig, master=graph_win)
        canvas.draw()
        canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
        plt.close(fig)

    def logout(self):
        self.root.destroy()

if __name__ == "__main__":
    root = tk.Tk()
    LoginPage(root)
    root.mainloop()
